In [ ]:
"""
=============================================================
FILE 44 — SUBGRAPH ARCHITECTURE
=============================================================

CONCEPTS TAUGHT
----------------
1. Subgraphs
2. Modular AI Systems
3. Parent Graphs
4. Child Graphs
5. Workflow Composition
6. Reusable AI Components
7. Enterprise Workflow Design
8. Nested Orchestration
9. Large Scale AI Systems
10. Graph Reusability

CORE IDEA
-----------
Large AI systems should be modular.

Instead of:
One giant workflow

Use:
Reusable subgraphs.

FLOW
-----
Parent Graph
   ↓
Subgraph A
Subgraph B
   ↓
Combine Outputs

REAL WORLD USE CASES
---------------------
- Enterprise AI platforms
- Large automation systems
- AI orchestration layers
- Multi-team workflows
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE MODEL
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):

    business_problem: str

    finance_analysis: str

    marketing_analysis: str

    final_strategy: str

# ============================================================
# STEP 5 — CREATE FINANCE SUBGRAPH
# ============================================================

def finance_node(state: State):

    response = llm.invoke(
        f"""
        Analyze this business problem
        from FINANCE perspective.

        PROBLEM:
        {state['business_problem']}
        """
    )

    return {
        "finance_analysis": response.content
    }

finance_builder = StateGraph(State)

finance_builder.add_node(
    "finance_node",
    finance_node
)

finance_builder.add_edge(
    START,
    "finance_node"
)

finance_builder.add_edge(
    "finance_node",
    END
)

finance_subgraph = finance_builder.compile()

# ============================================================
# STEP 6 — CREATE MARKETING SUBGRAPH
# ============================================================

def marketing_node(state: State):

    response = llm.invoke(
        f"""
        Analyze this business problem
        from MARKETING perspective.

        PROBLEM:
        {state['business_problem']}
        """
    )

    return {
        "marketing_analysis": response.content
    }

marketing_builder = StateGraph(State)

marketing_builder.add_node(
    "marketing_node",
    marketing_node
)

marketing_builder.add_edge(
    START,
    "marketing_node"
)

marketing_builder.add_edge(
    "marketing_node",
    END
)

marketing_subgraph = marketing_builder.compile()

# ============================================================
# STEP 7 — FINAL STRATEGY NODE
# ============================================================

def strategy_node(state: State):

    response = llm.invoke(
        f"""
        Create enterprise strategy using:

        FINANCE:
        {state['finance_analysis']}

        MARKETING:
        {state['marketing_analysis']}
        """
    )

    return {
        "final_strategy": response.content
    }

# ============================================================
# STEP 8 — PARENT GRAPH
# ============================================================

parent_builder = StateGraph(State)

parent_builder.add_node(
    "finance_subgraph",
    finance_subgraph
)

parent_builder.add_node(
    "marketing_subgraph",
    marketing_subgraph
)

parent_builder.add_node(
    "strategy_node",
    strategy_node
)

# ============================================================
# STEP 9 — DEFINE EDGES
# ============================================================

parent_builder.add_edge(
    START,
    "finance_subgraph"
)

parent_builder.add_edge(
    START,
    "marketing_subgraph"
)

parent_builder.add_edge(
    "finance_subgraph",
    "strategy_node"
)

parent_builder.add_edge(
    "marketing_subgraph",
    "strategy_node"
)

parent_builder.add_edge(
    "strategy_node",
    END
)

# ============================================================
# STEP 10 — COMPILE PARENT GRAPH
# ============================================================

graph = parent_builder.compile()

# ============================================================
# STEP 11 — VISUALIZE GRAPH
# ============================================================

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 12 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "business_problem":
        """
        AI transformation roadmap
        for a retail enterprise.
        """
    }
)

# ============================================================
# STEP 13 — PRINT RESULT
# ============================================================

print("\nFINAL STRATEGY\n")
print("=" * 60)

print(result["final_strategy"])